# Interaction-Ready Synthetic Data — Live Workshop

**Real2Sim for Embodied Data Generation**
*Generating Interaction-Ready Data for Long-Horizon Articulated Manipulation*

硬件路径：**CDNA3 (MI300/MI325, gfx942)** 单路径。范围：**只到 interaction-ready 数据生成**——
现场用 CAP 声明一个长程铰接体任务 `open → pick → place → close`（抽屉），经 rocRobo 无碰撞规划
跑出带交互语义的 LeRobot 数据集。本场**不做** train / eval。

> 按 cell 顺序执行即可。镜像已预装一切，产物落 `output/`。

## 0. 环境探测

确认两件事就绪：① 本容器能看到 gfx942 GPU；② rocRobo 无碰撞规划 sidecar（`rocrobo_dev`）可达。
（两个镜像由讲师预先起好。）

In [ ]:
import os, subprocess, torch

print("== GPU / ROCm ==")
print("torch:", torch.__version__, "| GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
print("HSA_OVERRIDE_GFX_VERSION:", os.environ.get("HSA_OVERRIDE_GFX_VERSION"))
print("ROCROBO_BACKEND:", os.environ.get("ROCROBO_BACKEND"))

print("\n== rocRobo sidecar ==")
r = subprocess.run(
    ["docker", "exec", "rocrobo_dev", "python", "-c", "import rocrobo; print('rocrobo_ok')"],
    capture_output=True, text=True,
)
print(r.stdout.strip() or r.stderr.strip() or "sidecar unreachable")

## 1. 上一场做到哪 → 这一场补什么

[上一场](https://github.com/PhysicalAI-AIM/Robot_synthetic_data_generation_workshop/tree/main)
已经证明 **合成数据 → VLA 训练 → 闭环评测**整条链路能在 AMD ROCm 上跑通，
但任务是**抓一个红方块**——单步 rigid pick，交互是平凡的。

这一场只补一个台阶：**能不能生成"和世界交互"的数据**。

| 维度 | 上一场（cube pick） | 这一场 |
|---|---|---|
| 任务 | 单步抓方块 | **长程 `open→pick→place→close`** |
| 物体 | 纯刚体 | **铰接体（抽屉，URDF 关节）** |
| 运动 | IK + 直线 | **无碰撞规划 + 段级重锚 + 接触段受控** |
| 数据语义 | state/action + 图像 | **+ 关节开度进 obs、段级重锚、接触相位** |

## 2. 交互任务从哪来：Real2Sim 资产

交互数据的前提是**有可交互的 sim 资产**：真实物体/场景 → 网格 / URDF（含关节、接触面）。
资产管线走 rocRecon（真实→可仿真资产）；本场直接用内置 objaverse 资产兜底
（drawer 抽屉柜 + die 方块）。重点在抽屉的 **URDF 关节**——它让"开 / 关"成为可交互动作。

## 3. CAP：~15 行声明一个交互任务

下面这个 scenario 就是作者侧的全部输入。CAP 把**任务意图**和**执行细节**解耦——
作者只声明四层：

- **scene**：放哪些物体（drawer + die）、锚点、相机
- **target**：放进抽屉的目标点（锚在抽屉的活动关节上，随实际开度移动）
- **task**：自然语言指令 + 成功判定（die 在抽屉内 且 抽屉关闭）
- **intents**：技能序列 `open → pick → place → close`

不同参与者改这一个文件就能声明各自的交互任务——这就是"声明式数据工厂"的入口。

> **怎么写 CAP？** 本场直接在 Jupyter 文件编辑器里改这个 `.py`——它在挂载进容器的仓库里，
> 存盘即对容器生效，不用"搬文件"。CAP 刻意做成 ~15 行可读 Python，**手改即可，不依赖本地 code agent**；
> 用 LLM/agent 自动生成 CAP 是 code-as-policy 的进阶玩法，属作者侧另一条线，不是现场机制。

In [ ]:
from pathlib import Path

SCENARIO = "scenarios/pick_place_into_drawer.py"
print(Path(SCENARIO).read_text())

### 把声明渲染成一张图（先看场景对不对）

跑完整 rollout 之前，先把刚声明的 scenario **静态渲染一帧**：build 场景 + 摆好物体 + Franka 归位，
确认 drawer / die 的摆位符合预期。这步也顺带付掉 `scene.build()` 的首次 kernel 编译（已缓存），
后面正式生成就更快。

In [ ]:
import subprocess, sys, glob
from pathlib import Path
from IPython.display import Image as IPyImage

# 跟随上面的 SCENARIO：改了哪个 scenario 就预览哪个。
SNAP_OUT = f"output/{Path(SCENARIO).stem}_snapshot"
cmd = [
    sys.executable, "scripts/datagen/snapshot_scenario.py",
    "--scenario", SCENARIO,
    "--out", SNAP_OUT,
    "--seed", "42",
]
print(">>", " ".join(cmd), "\n")
subprocess.run(cmd, check=True)

pngs = sorted(glob.glob(f"{SNAP_OUT}/*overview*.png"))
IPyImage(filename=pngs[0]) if pngs else None

### 🙋 你来改（秒级反馈，不用等生成）

快照只 build + 渲一帧，几秒出图。现在**你来声明**：在左侧文件树双击打开
`scenarios/pick_place_into_drawer.py`，改一行存盘，再**重跑上面那个快照 cell** 看变化。

几个可以试的（任选）：

- 把 `die` 的 `fixed_position` 里的 `y`（`0.32`）改成 `-0.15` → 方块换到抽屉另一侧
- 把 `asset="die_01"` 换成 `asset="apple_01"` → 换一个物体
- 把上面的 `SCENARIO` 改成 `scenarios/pick_one_of_three.py`，再跑快照 → 看桌面三选一布局

> 满意了再往下跑生成（生成慢，所以先在快照里把声明调对）。

## 4. 为什么交互需要无碰撞规划

简单 pick：IK + 一条直线就够。但一旦有**环境交互**——铰接抽屉、把物体放进容器、长程多段——
机械臂必须绕开抽屉柜体、在接触段受控移动，**直线必然撞**。

所以 `pick` / `place` 的运动由 **rocRobo 无碰撞规划**给出（跑在 ROCm 上）；
抓取候选由一个学习模型给出、按可行性筛选；抽屉的 `open` / `close` 走 drag-handle 原语。
下一步的生成会真正调用这条规划。

下面这段 **collision-blind vs collision-aware 分屏对比**（rocRobo `samples/demo_compare.py`
的预渲染产物，随 rocRobo 镜像自带）最直观：同一个目标，直线规划穿模/撞障，rocRobo 绕行避障。

In [1]:
from IPython.display import Video

# rocRobo samples/demo_compare.py 的预渲染对比视频，作为 workshop 素材直接放在 videos/。
Video("workshop/videos/rocrobo_compare.mp4", embed=True, width=720)

## 5. 现场生成一条交互 episode（hero）

一次脚本调用跑完整条 `open→pick→place→close`：rocRobo 规划每段无碰撞轨迹、
学习抓取选 grasp、按抽屉实时开度重锚目标点、渲染相机、导出 LeRobot 数据集 + 视频。

> CDNA3 无图形管线，相机走 CPU 软渲染，单条 **数十秒~分钟级**，属正常。现场只生成 1 条。

In [ ]:
import subprocess, sys, time

OUTPUT_DIR = "output/ws_drawer"
cmd = [
    sys.executable, "scripts/datagen/run_generated_scenario.py",
    "--scenario", "scenarios/pick_place_into_drawer.py",
    "--output-dir", OUTPUT_DIR,
    "--n-episodes", "1",
    "--seed", "42",
    "--grasp-planner", "auto",
    "--smoke",
]
print(">>", " ".join(cmd), "\n")
t0 = time.time()
subprocess.run(cmd, check=True)
print(f"\n[done] {time.time() - t0:.1f}s  ->  {OUTPUT_DIR}/")

## 6. 打开产出

播放刚生成的 episode 视频——完整的 `open→pick→place→close` 跑给你看。
（现场没跑完就回退到预生成视频。）

In [ ]:
import glob
from IPython.display import Video

vids = sorted(glob.glob("output/ws_drawer/videos/*.mp4"))
if not vids:  # 现场慢 / 没跑完：回退预生成兜底视频
    vids = sorted(glob.glob("workshop/videos/*.mp4"))
print("video:", vids[0] if vids else "(none)")
Video(vids[0], embed=True, width=640) if vids else None

## 7. 换一个 scenario 重生成（声明式数据工厂）

同一条流水线，换一个 ~15 行声明就能产出不同的交互数据。本场内置三个 demo scenario：

| scenario | 任务 | 类型 |
|---|---|---|
| `pick_place_into_drawer.py` | 开抽屉 → 抓 die → 放入 → 关 | 长程铰接体（hero） |
| `pick_place_onto_supporter.py` | 把苹果从上层架挪到下层架 | 无碰撞 pick-place（静态） |
| `pick_one_of_three.py` | 桌面随机摆 3 物，pick 苹果 | 单步 rigid pick |

改下面的 `SCENARIO` 选一个重跑即可（也可以直接进某个 `.py` 改一行——换物体 / 挪落点 / 改成功判定）。

In [ ]:
import subprocess, sys
from pathlib import Path

# 任选一个 demo scenario 重生成：
#   scenarios/pick_one_of_three.py          桌面三选一 pick（最快）
#   scenarios/pick_place_onto_supporter.py  上层→下层 无碰撞 pick-place
#   scenarios/pick_place_into_drawer.py     hero（同 §5）
SCENARIO = "scenarios/pick_one_of_three.py"

NAME = "ws_" + Path(SCENARIO).stem
cmd = [
    sys.executable, "scripts/datagen/run_generated_scenario.py",
    "--scenario", SCENARIO,
    "--output-dir", f"output/{NAME}",
    "--n-episodes", "1",
    "--seed", "1",
    "--grasp-planner", "auto",
    "--smoke",
]
print(">>", " ".join(cmd), "\n")
subprocess.run(cmd, check=True)

## 8. 小结

- **interaction-ready 的来源**：铰接交互（抽屉关节）+ 长程多段 + 无碰撞规划 + 接触语义，
  全部落进 LeRobot 数据集，可直接喂训练。
- **AMD / ROCm 工程故事**：rocRecon（资产）/ rocRobo（无碰撞规划）/ Genesis（物理 + 渲染）
  全跑在 ROCm 上，全程许可干净。
- 改一行声明就能换出新任务——这是一个跑在 AMD 上的**声明式交互数据工厂**。